# Synthesize ToM Data Using OpenRouter API

This notebook generates synthetic ToM questions for each Layer 3 cluster by:
1. Loading cluster summaries and failure examples
2. Selecting 2 random examples from each cluster
3. Filling the prompt template
4. Calling LLM via OpenRouter API 20 times to generate 10 questions per call (200 total per cluster)

## Expected Output Format

Each question generated will have:
- `story`: The narrative context
- `question`: The ToM reasoning question
- `option_a`, `option_b`, `option_c`, `option_d`: Four answer choices
- `correct_answer`: Single letter ("A", "B", "C", or "D") representing the human-correct answer

In [20]:
import os
import json
import pandas as pd
import random
import time
import csv
from openai import OpenAI
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Set random seed for reproducibility
random.seed(42)

# Thread-safe lock for progress bar updates
progress_lock = threading.Lock()

print("Libraries imported successfully")

Libraries imported successfully


## Configuration

In [45]:
# API Configuration
import re
with open(os.path.expanduser('~/.bashrc'), 'r') as f:
  bashrc_content = f.read()
match = re.search(r'export OPENROUTER_API_KEY=([^\s\n]+)', bashrc_content)
api_key = match.group(1) if match else None

client = OpenAI(
  api_key=api_key,
  base_url="https://openrouter.ai/api/v1"
)

MODEL = "openai/gpt-5"  # or other models available on OpenRouter

# Generation Configuration
# CLUSTERS_TO_PROCESS = [3, 4]  # File indices (table clusters 4 and 5)
CLUSTERS_TO_PROCESS = [8]
QUESTIONS_PER_BATCH = 10  # Generate 10 questions per API call

NUM_BATCHES = 10  # ← SET THIS VALUE (e.g., 20 for 200 questions, 1 for testing)

# Multithreading Configuration
MAX_WORKERS = 10  # ← Number of parallel API calls (adjust based on rate limits)

TOTAL_QUESTIONS_PER_CLUSTER = QUESTIONS_PER_BATCH * NUM_BATCHES

# Clusters to process (only clusters with objectivity > 5 and sufficient samples)
# Note: Table shows clusters 1-10, but files are indexed 0-9
# Table Cluster 4 (objectivity: 7) = File cluster_3.json
# Table Cluster 5 (objectivity: 10) = File cluster_4.json
# Table Cluster 9 (objectivity: 6) = File cluster_8.json - REMOVED (only 2 items)

# File paths
PROMPT_TEMPLATE_PATH = "data_synthesize_prompt.txt"
FAILURES_CSV_PATH = "failures/qwen2_5_7b_instruct_noCoT_qwen2_5_7b_instruct_noCoT_train_results_failures.csv"
CURRENT_INDEX_LOOKUP_PATH = "results/connecting_layers/current_index_lookup.json"
LAYER3_ANALYSIS_DIR = "results/cluster_analysis_layer3"
CONNECTING_LAYERS_DIR = "results/connecting_layers"
OUTPUT_DIR = "results/synthesized_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ Using OpenRouter API with model: {MODEL}")
print(f"✓ Will process clusters (0-indexed): {CLUSTERS_TO_PROCESS}")
print(f"  (Corresponding to table clusters: {[c+1 for c in CLUSTERS_TO_PROCESS]})")
print(f"✓ Generation: {NUM_BATCHES} batches × {QUESTIONS_PER_BATCH} questions = {TOTAL_QUESTIONS_PER_CLUSTER} per cluster")
print(f"✓ Multithreading: {MAX_WORKERS} parallel workers")
print(f"  Total API calls: {len(CLUSTERS_TO_PROCESS)} clusters × {NUM_BATCHES} batches = {len(CLUSTERS_TO_PROCESS) * NUM_BATCHES} calls")

✓ Output directory: results/synthesized_data
✓ Using OpenRouter API with model: openai/gpt-5
✓ Will process clusters (0-indexed): [8]
  (Corresponding to table clusters: [9])
✓ Generation: 10 batches × 10 questions = 100 per cluster
✓ Multithreading: 10 parallel workers
  Total API calls: 1 clusters × 10 batches = 10 calls


In [46]:
# Load prompt template
with open(PROMPT_TEMPLATE_PATH, 'r', encoding='utf-8') as f:
    prompt_template = f.read()

print(f"✓ Loaded prompt template ({len(prompt_template)} chars)")

✓ Loaded prompt template (3379 chars)


In [47]:
# Load failures CSV
failures_df = pd.read_csv(FAILURES_CSV_PATH)
print(f"✓ Loaded failures CSV: {len(failures_df)} rows")
print(f"  Columns: {list(failures_df.columns)}")

✓ Loaded failures CSV: 645 rows
  Columns: ['index', 'STORY', 'QUESTION', 'OPTION-A', 'OPTION-B', 'OPTION-C', 'OPTION-D', 'ANSWER', 'SOURCE_FILE', 'qwen2_5_7b_instruct_noCoT_full_response', 'qwen2_5_7b_instruct_noCoT_response', 'qwen2_5_7b_instruct_noCoT_truncation', 'qwen2_5_7b_instruct_noCoT_prompt_tokens', 'qwen2_5_7b_instruct_noCoT_output_tokens']


In [48]:
# Load current_index_lookup
with open(CURRENT_INDEX_LOOKUP_PATH, 'r', encoding='utf-8') as f:
    current_index_lookup = json.load(f)

# Convert string keys to int for easier lookup
current_index_lookup = {int(k): v for k, v in current_index_lookup.items()}

print(f"✓ Loaded current_index_lookup: {len(current_index_lookup)} entries")

✓ Loaded current_index_lookup: 645 entries


In [49]:
# Load Layer 3 cluster summaries (only for selected clusters)
layer3_summaries = {}
for i in CLUSTERS_TO_PROCESS:
    analysis_file = os.path.join(LAYER3_ANALYSIS_DIR, f"cluster_{i}_analysis.json")
    with open(analysis_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        layer3_summaries[i] = data['cluster_summary']

print(f"✓ Loaded {len(layer3_summaries)} Layer 3 cluster summaries for clusters: {CLUSTERS_TO_PROCESS}")

✓ Loaded 1 Layer 3 cluster summaries for clusters: [8]


In [50]:
# Load connecting layers to get current_indices for each cluster (only for selected clusters)
layer3_current_indices = {}
for i in CLUSTERS_TO_PROCESS:
    cluster_file = os.path.join(CONNECTING_LAYERS_DIR, f"layer3_cluster_{i}_complete.json")
    with open(cluster_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        layer3_current_indices[i] = data['all_current_indices']

print(f"✓ Loaded current_indices for {len(layer3_current_indices)} clusters")
for i in CLUSTERS_TO_PROCESS:
    print(f"  Cluster {i} (table cluster {i+1}): {len(layer3_current_indices[i])} items")

✓ Loaded current_indices for 1 clusters
  Cluster 8 (table cluster 9): 2 items


## Helper Functions

In [51]:
def get_example_data(current_idx):
    """
    Get all data for a specific current_index from failures CSV and lookup.
    
    current_index = row position in CSV (0-644)
    original_index = value in CSV's "index" column
    """
    # Get row directly using current_index as row position
    if current_idx >= len(failures_df):
        raise ValueError(f"current_index {current_idx} out of range (CSV has {len(failures_df)} rows)")
    
    row = failures_df.iloc[current_idx]
    
    # Get failed_summary from lookup
    lookup_data = current_index_lookup.get(current_idx)
    if not lookup_data:
        raise ValueError(f"current_index {current_idx} not found in lookup")
    
    return {
        'story': row['STORY'],
        'question': row['QUESTION'],
        'option_a': row['OPTION-A'],
        'option_b': row['OPTION-B'],
        'option_c': row['OPTION-C'],
        'option_d': row['OPTION-D'],
        'human_answer': row['ANSWER'],
        'model_answer': row['qwen2_5_7b_instruct_noCoT_response'],
        'reason': lookup_data['failed_summary']
    }

def fill_prompt_template(cluster_summary, example1_data, example2_data):
    """
    Fill the prompt template with cluster summary and two examples.
    """
    filled_prompt = prompt_template.format(
        cluster_summary=cluster_summary,
        example1_story=example1_data['story'],
        example1_question=example1_data['question'],
        example1_option_a=example1_data['option_a'],
        example1_option_b=example1_data['option_b'],
        example1_option_c=example1_data['option_c'],
        example1_option_d=example1_data['option_d'],
        example1_human_answer=example1_data['human_answer'],
        example1_model_answer=example1_data['model_answer'],
        example1_reason=example1_data['reason'],
        example2_story=example2_data['story'],
        example2_question=example2_data['question'],
        example2_option_a=example2_data['option_a'],
        example2_option_b=example2_data['option_b'],
        example2_option_c=example2_data['option_c'],
        example2_option_d=example2_data['option_d'],
        example2_human_answer=example2_data['human_answer'],
        example2_model_answer=example2_data['model_answer'],
        example2_reason=example2_data['reason']
    )
    return filled_prompt

def call_gpt(prompt, temperature=2, max_tokens=8000, max_retries=3, retry_delay=3):
    """
    Call LLM via OpenRouter API with the filled prompt.
    Retries up to max_retries times with retry_delay seconds between attempts.
    
    Adjust temperature here (0.0-2.0):
    - Lower (0.0-0.3): More deterministic
    - Medium (0.5-0.7): Balanced  
    - Higher (0.8-2.0): More creative
    
    Note: max_tokens set to 5K for ~10 questions
    Each question ~400 tokens × 10 = ~4K tokens needed
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are an expert in Theory of Mind reasoning and educational dataset creation."},
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                max_tokens=max_tokens,
                response_format={"type": "json_object"}
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                # Not the last attempt, wait and retry
                time.sleep(retry_delay)
                continue
            else:
                # Last attempt failed, raise the exception
                raise e

def process_batch(batch_num, filled_prompt):
    """
    Process a single batch - thread-safe function for parallel execution.
    Returns (batch_num, questions_list, error_message)
    """
    try:
        # Call OpenRouter API
        gpt_response = call_gpt(filled_prompt)
        
        # Parse JSON response (expecting {"questions": [...]})
        response_data = json.loads(gpt_response)
        
        # Validate response format
        if 'questions' not in response_data:
            return (batch_num, [], f"Response missing 'questions' array")
        
        questions = response_data['questions']
        
        # Validate count
        warning = None
        if len(questions) != QUESTIONS_PER_BATCH:
            warning = f"Got {len(questions)} questions (expected {QUESTIONS_PER_BATCH})"
        
        return (batch_num, questions, warning)
        
    except Exception as e:
        return (batch_num, [], str(e))

print("✓ Helper functions defined (with multithreading support)")

✓ Helper functions defined (with multithreading support)


In [52]:
# Generate synthetic data for selected clusters only (MULTITHREADED)
synthesized_data = {}

for cluster_id in CLUSTERS_TO_PROCESS:
    print(f"\n{'='*70}")
    print(f"Processing Cluster {cluster_id} (Table Cluster {cluster_id+1})")
    print(f"{'='*70}")
    
    # Get cluster summary
    cluster_summary = layer3_summaries[cluster_id]
    print(f"Cluster summary: {cluster_summary[:100]}...")
    
    # Get current_indices for this cluster
    current_indices = layer3_current_indices[cluster_id]
    print(f"Number of items in cluster: {len(current_indices)}")
    
    # Select 2 random examples (or all if only 2 or fewer)
    if len(current_indices) <= 2:
        selected_indices = current_indices
        print(f"Using all {len(selected_indices)} items as examples")
    else:
        selected_indices = random.sample(current_indices, 2)
        print(f"Randomly selected 2 examples: {selected_indices}")
    
    # Get example data
    try:
        example1_data = get_example_data(selected_indices[0])
        example2_data = get_example_data(selected_indices[1]) if len(selected_indices) > 1 else example1_data
        
        # Fill prompt template once (same for all batches)
        filled_prompt = fill_prompt_template(cluster_summary, example1_data, example2_data)
        
        # Generate questions in batches with MULTITHREADING
        all_questions = []
        errors = []
        warnings = []
        
        print(f"\nGenerating {TOTAL_QUESTIONS_PER_CLUSTER} questions in {NUM_BATCHES} batches of {QUESTIONS_PER_BATCH}...")
        print(f"Using {MAX_WORKERS} parallel workers...")
        
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all batch jobs
            futures = {
                executor.submit(process_batch, batch_num, filled_prompt): batch_num 
                for batch_num in range(NUM_BATCHES)
            }
            
            # Process results as they complete with progress bar
            with tqdm(total=NUM_BATCHES, desc=f"Cluster {cluster_id}", unit="batch") as pbar:
                for future in as_completed(futures):
                    batch_num, questions, message = future.result()
                    
                    if questions:
                        all_questions.extend(questions)
                        if message:  # Warning (got different count)
                            warnings.append(f"Batch {batch_num+1}: ⚠️ {message}")
                    else:
                        errors.append(f"Batch {batch_num+1}: ✗ {message}")
                    
                    pbar.update(1)
        
        # Print warnings and errors
        for w in warnings:
            print(f"  {w}")
        for e in errors:
            print(f"  {e}")
        
        print(f"\n✓ Successfully generated {len(all_questions)} total questions for cluster {cluster_id}")
        if errors:
            print(f"  ({len(errors)} batches failed)")
        
        # Add metadata to the response
        synthesized_data[cluster_id] = {
            'cluster_id': cluster_id,
            'table_cluster_id': cluster_id + 1,  # For reference
            'cluster_summary': cluster_summary,
            'example_indices': selected_indices,
            'num_questions': len(all_questions),
            'questions': all_questions
        }
        
        if all_questions:
            print(f"  First story preview: {all_questions[0].get('story', '')[:80]}...")
        
    except Exception as e:
        print(f"✗ Error processing cluster {cluster_id}: {str(e)}")
        import traceback
        traceback.print_exc()
        synthesized_data[cluster_id] = {"error": str(e)}

print(f"\n{'='*70}")
print(f"Synthesis complete: {len(synthesized_data)} clusters processed")
print(f"Clusters processed: {list(synthesized_data.keys())}")
total_q = sum(d.get('num_questions', 0) for d in synthesized_data.values() if 'num_questions' in d)
print(f"Total questions generated: {total_q}")
print(f"{'='*70}")


Processing Cluster 8 (Table Cluster 9)
Cluster summary: Across this cluster, the model repeatedly ignores the zero-sum, competitive framing and instead attr...
Number of items in cluster: 2
Using all 2 items as examples

Generating 100 questions in 10 batches of 10...
Using 10 parallel workers...


Cluster 8: 100%|██████████████████████████████████████████████| 10/10 [01:34<00:00,  9.42s/batch]


✓ Successfully generated 100 total questions for cluster 8
  First story preview: At the regional debate tournament, Jamie and Lila are in the same bracket compet...

Synthesis complete: 1 clusters processed
Clusters processed: [8]
Total questions generated: 100


In [53]:
# Create a flattened CSV with all questions (one row per question)
all_questions_rows = []

for cluster_id, data in synthesized_data.items():
    if 'error' not in data and 'questions' in data:
        cluster_summary = data.get('cluster_summary', '')
        for i, q in enumerate(data['questions']):
            all_questions_rows.append({
                'cluster_id': cluster_id,
                'question_index': i,
                'story': q.get('story', ''),
                'question': q.get('question', ''),
                'option_a': q.get('option_a', ''),
                'option_b': q.get('option_b', ''),
                'option_c': q.get('option_c', ''),
                'option_d': q.get('option_d', ''),
                'correct_answer': q.get('correct_answer', ''),
                'cluster_summary': cluster_summary
            })

# Save to CSV with proper quoting to handle special characters
if all_questions_rows:
    all_questions_df = pd.DataFrame(all_questions_rows)
    flattened_csv = os.path.join(OUTPUT_DIR, "r2_cluster9_100Q.csv")
    all_questions_df.to_csv(
        flattened_csv, 
        index=False, 
        encoding='utf-8',
        quoting=csv.QUOTE_ALL,  # Quote all fields to handle special characters
        escapechar='\\'  # Set escape character for safety
    )
    
    print(f"✓ Saved flattened CSV: {flattened_csv}")
    print(f"  Total rows: {len(all_questions_df)}")
    print(f"  Columns: {list(all_questions_df.columns)}")
    print(f"\nFirst few rows:")
    print(all_questions_df.head())
else:
    print("No questions to export")

✓ Saved flattened CSV: results/synthesized_data/r2_cluster9_100Q.csv
  Total rows: 100
  Columns: ['cluster_id', 'question_index', 'story', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'correct_answer', 'cluster_summary']

First few rows:
   cluster_id  question_index  \
0           8               0   
1           8               1   
2           8               2   
3           8               3   
4           8               4   

                                               story  \
0  At the regional debate tournament, Jamie and L...   
1  In a sales department, the top seller each qua...   
2  The drama club is casting the lead role, and o...   
3  At the regional science fair, only one project...   
4  Two seniors, Dana and Luis, are finalists for ...   

                                            question  \
0  What emotion does Lila most likely feel when s...   
1  What is the most plausible intention behind Pr...   
2  Given the situation, how is Carla most 